# Orthogonal Model Market-Hour Diagnostics (Latest Run)

This notebook auto-detects the latest orthogonal run in `runs/` and analyzes performance/prediction behavior by **market** and **hour** for:
- OOF predictions (`cv_oof.csv`)
- Test-set predictions (`submission.csv` + `data/test_for_participants.csv`)


In [3]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

ROOT = Path.cwd().parent
RUNS_DIR = ROOT / 'runs'
DATA_DIR = ROOT / 'data'

print(f'ROOT: {ROOT}')
print(f'RUNS_DIR: {RUNS_DIR}')
print(f'DATA_DIR: {DATA_DIR}')


ROOT: /Users/simonbisgaard/Desktop/nitor
RUNS_DIR: /Users/simonbisgaard/Desktop/nitor/runs
DATA_DIR: /Users/simonbisgaard/Desktop/nitor/data


In [15]:
RUN_ID_OVERRIDE: str | None = None  # e.g. '20260222-132125_orthogonal_tuned_2h'

def is_orthogonal_run_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    name = p.name.lower()
    has_stamp = len(name) >= 15 and name[8] == '-'
    return has_stamp and 'orthogonal' in name and (p / 'submission.csv').exists()

def find_latest_orthogonal_run(runs_dir: Path) -> Path:
    candidates = sorted([p for p in runs_dir.iterdir() if is_orthogonal_run_dir(p)], key=lambda x: x.name)
    if not candidates:
        raise FileNotFoundError('No orthogonal run with submission.csv found under runs/.')
    return candidates[-1]

if RUN_ID_OVERRIDE:
    run_dir = Path(RUN_ID_OVERRIDE)
    if not run_dir.is_absolute():
        run_dir = RUNS_DIR / RUN_ID_OVERRIDE
else:
    run_dir = find_latest_orthogonal_run(RUNS_DIR)

print(f'Using run_dir: {run_dir}')
print('Run files:')
for p in sorted(run_dir.iterdir()):
    print(f' - {p.name}')


Using run_dir: /Users/simonbisgaard/Desktop/nitor/runs/20260222-165709_orthogonal_tuned_2h_mh19_calibrated
Run files:
 - calibration_metadata.json
 - calibration_table.csv
 - row_adjustments.csv
 - submission.csv


In [16]:
def add_hour(df: pd.DataFrame, ts_col: str = 'delivery_start') -> pd.DataFrame:
    out = df.copy()
    out[ts_col] = pd.to_datetime(out[ts_col], errors='coerce')
    out['hour'] = out[ts_col].dt.hour
    return out

submission = pd.read_csv(run_dir / 'submission.csv')
test_meta = pd.read_csv(DATA_DIR / 'test_for_participants.csv', usecols=['id', 'market', 'delivery_start'])
train = pd.read_csv(DATA_DIR / 'train.csv', usecols=['market', 'delivery_start', 'target'])

oof_path = run_dir / 'cv_oof.csv'
oof = pd.read_csv(oof_path) if oof_path.exists() else pd.DataFrame()

metrics_path = run_dir / 'metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8')) if metrics_path.exists() else {}

test_pred = test_meta.merge(submission, on='id', how='left')
if test_pred['target'].isna().any():
    raise ValueError('NaNs in test predictions after merge.')

train = add_hour(train)
test_pred = add_hour(test_pred)
if not oof.empty:
    oof = add_hour(oof)

print(f'train rows: {len(train):,}')
print(f'test rows: {len(test_pred):,}')
print(f'oof rows: {len(oof):,}')
print('metrics keys:', sorted(metrics.keys()))


train rows: 132,608
test rows: 13,098
oof rows: 0
metrics keys: []


## OOF Market-Hour Diagnostics

In [17]:
if oof.empty:
    print('No cv_oof.csv found in this run. OOF diagnostics skipped.')
else:
    oof_diag = oof.copy()
    oof_diag['err'] = oof_diag['pred'] - oof_diag['target']
    oof_diag['abs_err'] = oof_diag['err'].abs()
    oof_diag['sq_err'] = oof_diag['err'] ** 2

    oof_mh = (
        oof_diag.groupby(['market', 'hour'], dropna=False)
        .agg(
            n=('pred', 'size'),
            target_mean=('target', 'mean'),
            pred_mean=('pred', 'mean'),
            bias=('err', 'mean'),
            mae=('abs_err', 'mean'),
            rmse=('sq_err', lambda s: float(np.sqrt(np.mean(s)))),
            p90_abs_err=('abs_err', lambda s: float(np.quantile(s, 0.90))),
        )
        .reset_index()
        .sort_values(['rmse', 'mae'], ascending=[False, False])
    )

    print('Worst market-hour cells by RMSE:')
    display(oof_mh.head(30))

    oof_market = (
        oof_diag.groupby('market', dropna=False)
        .agg(
            n=('pred', 'size'),
            bias=('err', 'mean'),
            mae=('abs_err', 'mean'),
            rmse=('sq_err', lambda s: float(np.sqrt(np.mean(s)))),
        )
        .reset_index()
        .sort_values('rmse', ascending=False)
    )
    print('OOF by market:')
    display(oof_market)


No cv_oof.csv found in this run. OOF diagnostics skipped.


In [18]:
if oof.empty:
    print('No OOF heatmaps because cv_oof.csv is missing.')
else:
    rmse_pivot = oof_mh.pivot(index='market', columns='hour', values='rmse')
    bias_pivot = oof_mh.pivot(index='market', columns='hour', values='bias')

    fig_rmse = px.imshow(
        rmse_pivot,
        aspect='auto',
        color_continuous_scale='Reds',
        title='OOF RMSE by Market x Hour'
    )
    fig_rmse.update_layout(height=380)
    fig_rmse.show()

    fig_bias = px.imshow(
        bias_pivot,
        aspect='auto',
        color_continuous_scale='RdBu',
        origin='lower',
        title='OOF Bias (pred - target) by Market x Hour'
    )
    fig_bias.update_layout(height=380)
    fig_bias.show()


No OOF heatmaps because cv_oof.csv is missing.


## Test-Set Market-Hour Prediction Diagnostics

In [8]:
test_mh = (
    test_pred.groupby(['market', 'hour'], dropna=False)
    .agg(
        n=('target', 'size'),
        pred_mean=('target', 'mean'),
        pred_median=('target', 'median'),
        pred_std=('target', 'std'),
        pred_p05=('target', lambda s: float(np.quantile(s, 0.05))),
        pred_p95=('target', lambda s: float(np.quantile(s, 0.95))),
        pred_min=('target', 'min'),
        pred_max=('target', 'max'),
    )
    .reset_index()
)

print('Test prediction summary by market-hour:')
display(test_mh.sort_values(['market', 'hour']).head(60))

test_market = (
    test_pred.groupby('market', dropna=False)['target']
    .agg(['size', 'mean', 'median', 'std', 'min', 'max'])
    .rename(columns={'size': 'n'})
    .reset_index()
    .sort_values('mean', ascending=False)
)
print('Test prediction summary by market:')
display(test_market)


Test prediction summary by market-hour:


,market,hour,n,pred_mean,pred_median,pred_std,pred_p05,pred_p95,pred_min,pred_max
0,Market A,0,91,81.664043,61.566574,53.232051,29.075560,202.617111,18.832343,223.016558
1,Market A,1,91,83.967604,65.155958,50.148156,34.206087,186.622842,29.908889,226.519966
2,Market A,2,91,91.454636,72.161787,55.248409,33.567802,203.622838,30.445812,245.138893
3,Market A,3,91,96.296601,80.474477,59.630934,34.382471,207.486273,30.302476,278.115062
4,Market A,4,91,103.261786,84.715751,64.116806,34.719615,229.109439,28.688206,277.955330
5,Market A,5,91,108.616823,85.714531,66.831987,34.378670,243.401159,28.893588,278.826105
6,Market A,6,91,112.623469,92.584867,67.268572,34.229668,247.553099,30.673821,269.755485
7,Market A,7,91,86.234747,73.408910,42.119573,32.864306,175.188162,19.153340,186.514719
8,Market A,8,91,36.748491,33.437136,19.925494,12.499389,80.800632,2.869752,104.382804
9,Market A,9,91,23.262997,23.493724,12.127996,4.881889,42.028372,-6.900919,65.781290


Test prediction summary by market:


,market,n,mean,median,std,min,max
0,Market A,2183,61.926999,43.407321,54.631310,-37.2350,301.059815
1,Market B,2183,26.916002,24.966518,21.929488,-27.8125,262.912508
4,Market E,2183,26.895856,24.480213,22.424328,-28.6710,265.243951
2,Market C,2183,24.892603,25.088498,19.890384,-62.9105,147.160936
3,Market D,2183,22.575630,23.014308,18.329502,-37.5470,201.659506
5,Market F,2183,19.868941,20.385688,17.172510,-37.5908,187.860652


In [9]:
test_mean_pivot = test_mh.pivot(index='market', columns='hour', values='pred_mean')
test_std_pivot = test_mh.pivot(index='market', columns='hour', values='pred_std')

fig_test_mean = px.imshow(
    test_mean_pivot,
    aspect='auto',
    color_continuous_scale='Viridis',
    title='Test Predicted Mean by Market x Hour'
)
fig_test_mean.update_layout(height=380)
fig_test_mean.show()

fig_test_std = px.imshow(
    test_std_pivot,
    aspect='auto',
    color_continuous_scale='Magma',
    title='Test Predicted Std by Market x Hour'
)
fig_test_std.update_layout(height=380)
fig_test_std.show()


## Train-vs-Test Level Shift by Market-Hour

In [11]:
train_mh = (
    train.groupby(['market', 'hour'], dropna=False)['target']
    .agg(
        train_n='size',
        train_mean='mean',
        train_median='median',
        train_std='std',
        train_p05=lambda s: float(np.quantile(s, 0.05)),
        train_p95=lambda s: float(np.quantile(s, 0.95)),
    )
    .reset_index()
)

shift = test_mh.merge(train_mh, on=['market', 'hour'], how='left')
shift['mean_shift_test_minus_train'] = shift['pred_mean'] - shift['train_mean']
shift['mean_shift_z'] = shift['mean_shift_test_minus_train'] / (shift['train_std'] + 1e-6)
shift['p95_excess_vs_train_p95'] = shift['pred_p95'] - shift['train_p95']

print('Largest absolute mean-shift (test pred mean vs train target mean):')
display(
    shift.assign(abs_mean_shift=lambda d: d['mean_shift_test_minus_train'].abs())
    .sort_values('abs_mean_shift', ascending=False)
    .head(40)
)


Largest absolute mean-shift (test pred mean vs train target mean):


,market,hour,n,pred_mean,pred_median,pred_std,pred_p05,pred_p95,pred_min,pred_max,train_n,train_mean,train_median,train_std,train_p05,train_p95,mean_shift_test_minus_train,mean_shift_z,p95_excess_vs_train_p95,abs_mean_shift
115,Market E,19,91,-12.363931,-24.905130,28.447450,-28.671000,32.221430,-28.671000,169.792888,974,113.294569,32.2145,452.634688,-1.36365,245.46560,-125.658500,-0.277616,-213.244170,125.658500
43,Market B,19,91,-11.272898,-23.557546,28.270286,-27.812500,29.574192,-27.812500,172.625779,974,112.129171,31.4710,446.084788,-1.29040,255.14930,-123.402069,-0.276634,-225.575108,123.402069
19,Market A,19,91,16.535766,15.366571,42.849743,-37.235000,97.050926,-37.235000,185.121769,974,138.823419,53.2360,450.238071,0.28095,408.30185,-122.287653,-0.271607,-311.250924,122.287653
91,Market D,19,91,-32.839986,-37.547000,14.706938,-37.547000,-12.725697,-37.547000,73.011532,974,82.174744,30.8830,332.612450,-28.32895,189.18070,-115.014730,-0.345792,-201.906397,115.014730
67,Market C,19,91,-34.981026,-39.257582,26.567310,-62.910500,3.735605,-62.910500,111.299125,974,70.195071,30.4865,307.573712,-18.67605,138.26350,-105.176097,-0.341954,-134.527895,105.176097
139,Market F,19,91,-30.709062,-37.590800,18.061734,-37.590800,-4.482425,-37.590800,84.998894,656,43.304294,25.4290,169.256204,-28.25675,111.38375,-74.013356,-0.437286,-115.866175,74.013356
6,Market A,6,91,112.623469,92.584867,67.268572,34.229668,247.553099,30.673821,269.755485,974,62.661852,30.0745,112.084195,1.63325,245.27120,49.961616,0.445751,2.281899,49.961616
5,Market A,5,91,108.616823,85.714531,66.831987,34.378670,243.401159,28.893588,278.826105,974,63.274244,27.5635,107.639016,-0.42640,297.47390,45.342579,0.421247,-54.072741,45.342579
4,Market A,4,91,103.261786,84.715751,64.116806,34.719615,229.109439,28.688206,277.955330,974,62.051967,24.7870,109.747529,-2.13715,303.34410,41.209819,0.375497,-74.234661,41.209819
3,Market A,3,91,96.296601,80.474477,59.630934,34.382471,207.486273,30.302476,278.115062,974,57.442518,23.9635,102.574836,-3.56570,274.08290,38.854082,0.378788,-66.596627,38.854082


In [12]:
shift_pivot = shift.pivot(index='market', columns='hour', values='mean_shift_z')
fig_shift = px.imshow(
    shift_pivot,
    aspect='auto',
    color_continuous_scale='RdBu',
    origin='lower',
    title='Test-vs-Train Mean Shift Z-score by Market x Hour'
)
fig_shift.update_layout(height=380)
fig_shift.show()


## OOF-vs-Test Prediction Behavior (by Market-Hour)

In [13]:
if oof.empty:
    print('No OOF data available for OOF-vs-Test comparison.')
else:
    oof_pred_mh = (
        oof.groupby(['market', 'hour'], dropna=False)['pred']
        .agg(oof_pred_mean='mean', oof_pred_std='std', oof_n='size')
        .reset_index()
    )

    compare = test_mh.merge(oof_pred_mh, on=['market', 'hour'], how='left')
    compare['pred_mean_delta_test_minus_oof'] = compare['pred_mean'] - compare['oof_pred_mean']
    compare['pred_std_delta_test_minus_oof'] = compare['pred_std'] - compare['oof_pred_std']

    print('Largest OOF vs test mean prediction shifts:')
    display(
        compare.assign(abs_mean_delta=lambda d: d['pred_mean_delta_test_minus_oof'].abs())
        .sort_values('abs_mean_delta', ascending=False)
        .head(40)
    )


Largest OOF vs test mean prediction shifts:


,market,hour,n,pred_mean,pred_median,pred_std,pred_p05,pred_p95,pred_min,pred_max,oof_pred_mean,oof_pred_std,oof_n,pred_mean_delta_test_minus_oof,pred_std_delta_test_minus_oof,abs_mean_delta
91,Market D,19,91,-32.839986,-37.547000,14.706938,-37.547000,-12.725697,-37.547000,73.011532,132.325272,216.817987,28,-165.165257,-202.111049,165.165257
67,Market C,19,91,-34.981026,-39.257582,26.567310,-62.910500,3.735605,-62.910500,111.299125,129.839263,227.473317,28,-164.820289,-200.906007,164.820289
115,Market E,19,91,-12.363931,-24.905130,28.447450,-28.671000,32.221430,-28.671000,169.792888,144.624958,221.951712,28,-156.988889,-193.504262,156.988889
43,Market B,19,91,-11.272898,-23.557546,28.270286,-27.812500,29.574192,-27.812500,172.625779,143.226355,216.474959,28,-154.499253,-188.204674,154.499253
139,Market F,19,91,-30.709062,-37.590800,18.061734,-37.590800,-4.482425,-37.590800,84.998894,116.247663,226.280506,28,-146.956724,-208.218773,146.956724
19,Market A,19,91,16.535766,15.366571,42.849743,-37.235000,97.050926,-37.235000,185.121769,159.178842,218.265548,28,-142.643077,-175.415805,142.643077
92,Market D,20,91,19.577263,16.885459,18.828285,-6.614550,53.575875,-20.344619,74.463476,50.725957,18.201532,28,-31.148694,0.626753,31.148694
18,Market A,18,91,86.682110,75.863516,43.074235,28.582794,166.092195,17.957953,198.270091,57.400146,22.201654,28,29.281964,20.872581,29.281964
44,Market B,20,91,38.595592,38.485669,24.265009,1.188229,82.941276,-6.870905,116.439491,66.070283,20.569365,28,-27.474691,3.695644,27.474691
17,Market A,17,91,60.128042,48.081631,50.356505,8.308482,161.516898,-3.714348,301.059815,36.818769,12.430483,28,23.309273,37.926022,23.309273


## Tuning Summary (if available)

In [14]:
tune_json_path = run_dir / 'hparam_tuning.json'
tune_trials_path = run_dir / 'hparam_tuning_trials.csv'

if tune_json_path.exists():
    tune = json.loads(tune_json_path.read_text(encoding='utf-8'))
    print('Best trial:', tune.get('best_trial_number'))
    print('Best CV RMSE:', tune.get('best_value'))
    print('Trials completed:', tune.get('n_trials_completed'))
else:
    print('No hparam_tuning.json found.')

if tune_trials_path.exists():
    tune_trials = pd.read_csv(tune_trials_path)
    display(tune_trials.sort_values('cv_rmse').head(20))
else:
    print('No hparam_tuning_trials.csv found.')


Best trial: 11
Best CV RMSE: 55.1668878211055
Trials completed: 13


,trial,cv_rmse,elapsed_seconds,oof_coverage,spike_power,uplift_clip_quantile
11,11,55.166888,633.516749,0.040379,0.760230,0.950181
8,8,55.614338,655.267615,0.040379,0.839654,0.993964
12,12,56.480660,638.744183,0.040379,0.754457,0.951564
7,7,64.391871,484.283963,0.040379,1.181170,0.959139
9,9,65.089456,583.063350,0.040379,1.768269,0.961625
1,1,66.298191,469.773735,0.040379,1.138217,0.967952
3,3,69.122497,672.441541,0.040379,1.858367,0.959737
6,6,70.304270,446.422566,0.040379,1.445873,0.964743
5,5,71.142474,576.876744,0.040379,2.038838,0.976428
10,10,71.492441,673.334258,0.040379,0.752021,0.994660
